On utilise depuis peu le bloc de colonnes B_... de la table borrowers pour gérer les contentieux.
Ce bloc n'est plus accessible depuis assez longtemps.

Questions :
- on sait que ce bloc contient des informations autres que celles relatives aux contentieux => les identifier, pour éventuellement les supprimer ou les récuperer dans un autre bloc
- une fois que l'adhérent n'est plus en contentieux, les infos relatives au contentieux doivent être supprimées. Cela n'a jamais été fait, mettre au point une méthode d'analyse pour identifier les cartes concernées.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime
from datetime import date

from kiblib.utils.db import DbConn

import requests
import json

In [2]:
def mod_borrower(userid, data):
    api_url = "http://cataloguekoha.ntrbx.local/cgi-bin/koha/rest.pl/user"
    url = f"{api_url}/{userid}"
    data = json.dumps(data)
    data2mod = f"data={data}"
    print(data2mod)
    response = requests.put(url, data=data2mod)
    print(f"{userid} : {response.content}")

In [3]:
db_conn = DbConn().create_engine()

# Préparation des données
- on part de la table borrowers pour créer un dataframe "adherents"
- création du dataframe "adh_bloc_ctx" : on ne garde du dataframe "adherents" que les colonnes permettant de bien caractériser l'adhérent (âge, type carte, site, date inscription, date expiration) et les colonnes du bloc B_...
- on ajoute à "adh_bloc_ctx" une colonne "contentieux", avec une valeur vrai/faux, via une jointure sur un dataframe reprenant les adhérents en contentieux.

In [17]:
#query = """SELECT * 
#FROM koha_prod.borrowers b"""
#adherents = pd.read_sql(query, db_conn)

adherents = pd.read_json("http://cataloguekoha.ntrbx.local/cgi-bin/koha/rest.pl/user")
len(adherents)

32855

In [31]:
adh_bloc_ctx = adherents[['borrowernumber', 'cardnumber', 'userid', 'B_address', 'B_address2', 'B_city', 'B_state',
       'B_zipcode', 'B_country', 'contactnote', 'dateofbirth',
       'branchcode', 'categorycode', 'dateenrolled', 'dateexpiry']]
# on exclut les cartes de collectivités
adh_bloc_ctx = adh_bloc_ctx[adh_bloc_ctx['categorycode'].isin(["CSVT", "CSLT", "BIBL", "MEDB", "MEDA", "MEDC", "MEDP", "COLD", "COLI"])]

In [32]:
# on crée un dataframe avec les adhérents en contentieux
query = """SELECT borrowernumber, DATE(date_due) as date_due
FROM koha_prod.issues i
WHERE date_due + INTERVAL 90 DAY <= CURDATE()
ORDER BY borrowernumber, date_due DESC"""
contentieux = pd.read_sql(query, db_conn)
contentieux['contentieux'] = True
contentieux = contentieux.drop_duplicates(subset='borrowernumber')

adh_bloc_ctx = adh_bloc_ctx.merge(contentieux, how='left', on='borrowernumber')
adh_bloc_ctx.loc[adh_bloc_ctx['contentieux'].isna(), 'contentieux'] = False

adh_bloc_ctx['contentieux'].value_counts()

False    29879
True      2368
Name: contentieux, dtype: int64

In [20]:
ctx2 = adh_bloc_ctx[adh_bloc_ctx['contentieux'] == True]
ctx2['year'] = ctx2['date_due'].astype(str).str[0:4]
ctx2['year'].value_counts().reset_index().sort_values(by='index')

/tmp/ipykernel_16430/1697610370.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ctx2['year'] = ctx2['date_due'].astype(str).str[0:4]


,index,year
2,2018,344
3,2019,316
6,2020,254
7,2021,166
5,2022,269
4,2023,313
1,2024,358
0,2025,571


In [27]:
# pour chaque champs, on a parfois des valeurs vides : on les passe en np.nan pour faciliter les requêtes
for c in adh_bloc_ctx.columns:
    adh_bloc_ctx.loc[adh_bloc_ctx[c] == '', c] = np.nan

# Gestion : correction des données
On supprime les infos du bloc contentieux pour les personnes qui ne sont pas en contentieux.

In [28]:
# pour plus de facilité, on peut renommer les champs dédiés à la gestion des contentieux
adh_bloc_ctx_lib = adh_bloc_ctx.rename(columns={"B_address":"date_appel_telephonique",
                              "B_address2":"resultat_appel",
                              "B_city":"date_demande_creation_tiers",
                              "B_state":"numero_titre_recette",
                              "B_zipcode":"numero_tiers_GF",
                              "B_country":"date_creation_titre_recette",
                              "contactnote":"documents_a_payer"})

On ne garde que les lignes pour lesquelles le bloc comprend des données.

In [29]:
adh_bloc_ctx_lib_with_data = adh_bloc_ctx_lib[(~adh_bloc_ctx_lib['date_appel_telephonique'].isna())
                                    | (~adh_bloc_ctx_lib['resultat_appel'].isna())
                                    | (~adh_bloc_ctx_lib['date_demande_creation_tiers'].isna())
                                    | (~adh_bloc_ctx_lib['numero_titre_recette'].isna())
                                    | (~adh_bloc_ctx_lib['numero_tiers_GF'].isna())
                                    | (~adh_bloc_ctx_lib['date_creation_titre_recette'].isna())
                                    | (~adh_bloc_ctx_lib['documents_a_payer'].isna())]
len(adh_bloc_ctx_lib_with_data)
adh_bloc_ctx_lib_with_data.to_csv("adh_contentieux.csv", index=False)

pour chaque colonne contentieux, les valeurs admises sont les suivantes :
- date_appel_telephonique : date au format AAAA-MM-JJ
- resultat_appel : ['usager joint', 'message vocal', 'téléphone inconnu']
- date_demande_creation_tiers : date au format AAAA-MM-JJ
- numero_titre_recette : numéro
- numero_tiers_GF : nombre
- date_creation_titre_recette : date au format AAAA-MM-JJ
- documents_a_payer : texte

In [24]:
# on identifie les lignes qui posent pb
# 
adh_bloc_ctx_lib_with_data_pb = adh_bloc_ctx_lib_with_data[
    ( 
        (~adh_bloc_ctx_lib_with_data['date_appel_telephonique'].astype(str).str.match("\d{4}-\d{2}-\d{2}"))
      & (adh_bloc_ctx_lib_with_data['date_appel_telephonique'].notna()) 
    )
    
      | (  
        (~adh_bloc_ctx_lib_with_data['resultat_appel'].isin(['usager joint', 'message vocal', 'téléphone inconnu']))
      & (adh_bloc_ctx_lib_with_data['resultat_appel'].notna()) 
    )
    
  | (  
        (~adh_bloc_ctx_lib_with_data['numero_titre_recette'].astype(str).str.match("\d"))
      & (adh_bloc_ctx_lib_with_data['numero_titre_recette'].notna()) 
    )
    
  | (  
        (~adh_bloc_ctx_lib_with_data['numero_tiers_GF'].astype(str).str.match("\d"))
      & (adh_bloc_ctx_lib_with_data['numero_tiers_GF'].notna()) 
    )  
    
  | (  
        (~adh_bloc_ctx_lib_with_data['date_demande_creation_tiers'].astype(str).str.match("\d{4}-\d{2}-\d{2}"))
      & (adh_bloc_ctx_lib_with_data['date_demande_creation_tiers'].notna()) 
    )
    
  | (  
        (~adh_bloc_ctx_lib_with_data['date_creation_titre_recette'].astype(str).str.match("\d{4}-\d{2}-\d{2}"))
      & (adh_bloc_ctx_lib_with_data['date_creation_titre_recette'].notna()) 
    )
]
len(adh_bloc_ctx_lib_with_data_pb)

0

On isole les lignes pour lesquelles il n'y a pas de contentieux.

In [25]:
adh_bloc_ctx_lib_with_data_without_ctx = adh_bloc_ctx_lib_with_data[adh_bloc_ctx_lib_with_data['contentieux'] == False]
len(adh_bloc_ctx_lib_with_data_without_ctx)

181

In [26]:
adh_bloc_ctx_lib_with_data_without_ctx[~adh_bloc_ctx_lib_with_data_without_ctx['date_appel_telephonique'].isna()]

,borrowernumber,cardnumber,userid,date_appel_telephonique,resultat_appel,date_demande_creation_tiers,numero_titre_recette,numero_tiers_GF,date_creation_titre_recette,documents_a_payer,dateofbirth,branchcode,categorycode,dateenrolled,dateexpiry,date_due,contentieux


In [15]:
ctx2clean = adh_bloc_ctx_lib_with_data_without_ctx[~adh_bloc_ctx_lib_with_data_without_ctx['date_appel_telephonique'].isna()]
userid2clean = ctx2clean['userid'].tolist()

In [16]:
data = {"B_address": None,
        "B_address2": None,
        "contactnote": None}
#        "B_city": None,
#        "B_state": None,
#        "B_zipcode": None,
#        "B_country": None,
        
for userid in userid2clean[:500]:
    mod_borrower(userid, data)

data={"B_address": null, "B_address2": null, "contactnote": null}
X0001592830 : b'{\n   "modified_fields" : {},\n   "success" : true\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002039459 : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
elavieville : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
Melan : b'{\n   "modified_fields" : {},\n   "success" : true\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0001945249 : b'{\n   "modified_fields" : {},\n   "success" : true\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002288185 : b'{\n   "modified_fields" : {},\n   "success" : true\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
cgoncalves : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_addres

X0002782850 : b'{\n   "modified_fields" : {},\n   "success" : true\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002675442 : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002733074 : b'{\n   "modified_fields" : {},\n   "success" : true\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002328577 : b'{\n   "modified_fields" : {},\n   "success" : true\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002284088 : b'{\n   "modified_fields" : {},\n   "success" : true\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002643540 : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002782416 : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002305950 : b'{\n   "modi

X0002832876 : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002576312 : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002289526 : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002856834 : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002590622 : b'{\n   "modified_fields" : {},\n   "success" : true\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002591186 : b'{\n   "modified_fields" : {},\n   "success" : true\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002591957 : b'{\n   "modified_fields" : {},\n   "success" : true\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002743011 : b'{\n   "succ

X0002804217 : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002781938 : b'{\n   "modified_fields" : {},\n   "success" : true\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002823195 : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002822174 : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002799957 : b'{\n   "modified_fields" : {},\n   "success" : true\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002800134 : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002795843 : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002785967 : b'{\n   "modi

X0002861890 : b'{\n   "success" : true,\n   "modified_fields" : {}\n}\n'
data={"B_address": null, "B_address2": null, "contactnote": null}
X0002867908 : b'{\n   "modified_fields" : {},\n   "success" : true\n}\n'


On isole les lignes pour lesquelles il y a contentieux.

In [15]:
adh_bloc_ctx_lib_with_data_with_ctx = adh_bloc_ctx_lib_with_data[adh_bloc_ctx_lib_with_data['contentieux'] == True]
len(adh_bloc_ctx_lib_with_data_with_ctx)

2284

In [37]:
adh_bloc_ctx_lib_with_data_with_ctx['date_due_'] = pd.to_datetime(adh_bloc_ctx_lib_with_data_with_ctx['date_due'])
adh_bloc_ctx_lib_with_data_with_ctx['date_appel_telephonique_'] = pd.to_datetime(adh_bloc_ctx_lib_with_data_with_ctx['date_appel_telephonique'])

In [43]:
adh_bloc_ctx_lib_with_data_with_ctx.loc[
    adh_bloc_ctx_lib_with_data_with_ctx['date_due_'] > adh_bloc_ctx_lib_with_data_with_ctx['date_appel_telephonique_'],
    'ctx_statut'
] = 'nouveau'
adh_bloc_ctx_lib_with_data_with_ctx['newctx_appel'] = (adh_bloc_ctx_lib_with_data_with_ctx['date_due_'] - adh_bloc_ctx_lib_with_data_with_ctx['date_appel_telephonique_']).dt.days

In [39]:
adh_bloc_ctx_lib_with_data_with_ctx['ctx_statut'].value_counts()

nouveau    351
Name: ctx_statut, dtype: int64

In [45]:
new_ctx = adh_bloc_ctx_lib_with_data_with_ctx[adh_bloc_ctx_lib_with_data_with_ctx['ctx_statut'] == 'nouveau']
new_ctx

,borrowernumber,cardnumber,userid,date_appel_telephonique,resultat_appel,date_demande_creation_tiers,numero_titre_recette,numero_tiers_GF,date_creation_titre_recette,documents_a_payer,...,branchcode,categorycode,dateenrolled,dateexpiry,date_due,contentieux,date_due_,date_appel_telephonique_,ctx_statut,newctx_appel
176,2030.0,X0002587394,X0002587394,2021-10-22,usager joint,2025-08-19,NaN,201254,NaN,C1400006532 - 9.90\r\nC1400009101 - 22.00,...,MED,BIBL,2022-10-25,2025-01-05,2024-02-16,True,2024-02-16,2021-10-22,nouveau,847.0
213,2418.0,X0002349756,X0002349756,2023-07-15,téléphone inconnu,NaN,NaN,NaN,NaN,NaN,...,MED,MEDC,2007-02-28,2026-05-06,2025-07-15,True,2025-07-15,2023-07-15,nouveau,731.0
227,2577.0,X0002039459,X0002039459,2023-09-19,message vocal,NaN,NaN,NaN,NaN,NaN,...,BUS,BIBL,2008-08-28,2026-02-05,2025-05-31,True,2025-05-31,2023-09-19,nouveau,620.0
268,2973.0,X0002716022,X0002716022,2022-11-05,téléphone inconnu,NaN,NaN,NaN,NaN,NaN,...,MED,BIBL,2010-07-24,2025-02-21,2024-05-16,True,2024-05-16,2022-11-05,nouveau,558.0
271,3020.0,X0002550497,X0002550497,2024-09-19,message vocal,NaN,NaN,NaN,NaN,NaN,...,MED,BIBL,2006-08-09,2026-01-19,2025-07-12,True,2025-07-12,2024-09-19,nouveau,296.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29080,84858.0,X0002818702,X0002818702,2025-06-13,usager joint,NaN,NaN,NaN,NaN,NaN,...,MED,BIBL,2025-03-23,2026-03-23,2025-07-09,True,2025-07-09,2025-06-13,nouveau,26.0
29404,85183.0,X0002814919,X0002814919,2023-02-03,message vocal,2023-09-20,20230138,100168,2023-09-21,C2400002345 - 33.00,...,MED,BIBL,2025-04-11,2026-04-11,2025-06-10,True,2025-06-10,2023-02-03,nouveau,858.0
29430,85209.0,X0002814520,X0002814520,2025-02-06,message vocal,NaN,NaN,NaN,NaN,NaN,...,MED,BIBL,2025-04-15,2026-04-15,2025-05-06,True,2025-05-06,2025-02-06,nouveau,89.0
30525,86318.0,X0002851853,X0002851853,2021-05-15,message vocal,NaN,NaN,NaN,NaN,NaN,...,MED,BIBL,2025-06-25,2026-06-25,2025-07-16,True,2025-07-16,2021-05-15,nouveau,1523.0


In [47]:
new_ctx[new_ctx['newctx_appel'] < 90].to_excel("test_ctx.xlsx")